# Day 30: ARIMA/GARCH 建模实战

**日期**: 2026-08-08 | **阶段**: 时序建模 | **时长**: 4-5小时

## 今日目标
- 亲手模拟 AR(1)/AR(2)/MA(1)/MA(3)，恢复参数验证理解
- 理解 PACF 定 AR 阶数、ACF 定 MA 阶数的直觉
- 用 AIC 网格搜索 ARMA/ARIMA 最优阶数
- GARCH 建模波动率聚集
- 改参数对比，建立直觉

## 核心概念速查

| 模型 | 公式 | 定阶方法 |
|------|------|----------|
| AR(p) | x_t = Σφ_i·x_{t-i} + ε_t | PACF 在 p 阶后截尾 |
| MA(q) | x_t = μ + Σθ_i·ε_{t-i} + ε_t | ACF 在 q 阶后截尾 |
| ARMA(p,q) | AR + MA 合并 | AIC 网格搜索 |
| ARIMA(p,d,q) | d次差分 + ARMA | AIC 网格搜索 |
| GARCH(p,q) | σ²_t = ω + αε²_{t-1} + βσ²_{t-1} | 对残差平方的 ACF/PACF |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Hiragino Sans GB', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import warnings
warnings.filterwarnings('ignore')

import akshare as ak
import statsmodels.api as sm
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import jarque_bera
from arch import arch_model
from scipy import stats

# ====== 数据获取 ======
def safe_fetch(symbol, start='20240101', end='20250808'):
    code = symbol.zfill(6)
    prefix = 'sh' if code.startswith(('6','9')) else 'sz'
    full = f'{prefix}{code}'
    df = ak.stock_zh_a_daily(symbol=full, adjust='qfq')
    df = df.rename(columns={
        'date':'日期','open':'开盘','close':'收盘','high':'最高','low':'最低',
        'volume':'成交量','amount':'成交额','outstanding_share':'流通市值','turnover':'换手率'
    })
    df['日期'] = pd.to_datetime(df['日期'])
    df = df[(df['日期'] >= start) & (df['日期'] <= end)]
    return df

print('✅ 环境就绪')

---
## 第一部分：AR(p) — 自回归模型

### 核心公式
$$x_t = \phi_1 x_{t-1} + \phi_2 x_{t-2} + ... + \phi_p x_{t-p} + \varepsilon_t$$

**直觉**：今天 = 过去 p 天的加权平均 + 随机噪音。

**关键**：PACF（偏自相关）在 p 阶后截尾 → 用来判断 AR 的阶数 p。

### 1.1 模拟 AR(1)：φ=0.6

先模拟一个已知参数的 AR(1)，看 PACF 图是否能在 lag=1 处截尾。

In [ ]:
np.random.seed(42)
n = 500
phi1 = 0.6  # 真实参数

# 手动模拟 AR(1): x_t = 0.6 * x_{t-1} + ε_t
x_ar1 = np.zeros(n)
w = np.random.normal(0, 1, n)
for t in range(1, n):
    x_ar1[t] = phi1 * x_ar1[t-1] + w[t]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0,0].plot(x_ar1, linewidth=0.5)
axes[0,0].set_title(f'AR(1) φ={phi1} 模拟序列')
axes[0,0].grid(True, alpha=0.3)

plot_acf(x_ar1, lags=30, ax=axes[0,1])
axes[0,1].set_title('ACF（缓慢衰减=AR特征）')

plot_pacf(x_ar1, lags=30, ax=axes[1,0], method='ywm')
axes[1,0].set_title('PACF（lag=1后截尾→AR(1)✅）')

axes[1,1].hist(x_ar1, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[1,1].set_title('分布（近似正态）')
plt.tight_layout()
plt.savefig('/Users/macbookpro/Desktop/股/day30/01_AR1模拟.png', dpi=150, bbox_inches='tight')
plt.show()

print('📊 观察：PACF 在 lag=1 处有一个显著的柱，之后全部在蓝色区域内')
print('   → 这说明 AR(1) 模型是合适的')
print('   → PACF 第k个柱子 = 排除中间变量后，x_t和x_{t-k}的"净"相关性')

In [ ]:
# 用 statsmodels 拟合 AR 模型，看能否恢复 φ=0.6
model_ar1 = AutoReg(x_ar1, lags=1, trend='n').fit()
print(f'真实 φ = 0.6')
print(f'估计 φ = {model_ar1.params[0]:.4f}')
print(f'误差   = {abs(model_ar1.params[0] - 0.6):.4f}')
print()

# 残差检查
resid = model_ar1.resid
_, pval = jarque_bera(resid)
print(f'残差正态性检验 p={pval:.3f} {"(接近正态)✅" if pval > 0.1 else "(非正态)⚠️"}')

### 1.2 模拟 AR(2)：φ₁=0.666, φ₂=-0.333

第二个参数是负的 → 有均值回归的味道（涨多了会跌）。

In [ ]:
np.random.seed(42)
n = 500
phi = np.array([0.666, -0.333])

# 用 statsmodels 内置方法生成 AR(2)
ar = np.r_[1, -phi]  # statsmodels 的 AR 系数需要取负
ma = np.r_[1, 0.]
x_ar2 = sm.tsa.arma_generate_sample(ar=ar, ma=ma, nsample=n, burnin=100)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(x_ar2, linewidth=0.5)
axes[0].set_title('AR(2) φ₁=0.666 φ₂=-0.333'); axes[0].grid(True, alpha=0.3)
plot_acf(x_ar2, lags=30, ax=axes[1])
axes[1].set_title('ACF')
plot_pacf(x_ar2, lags=30, ax=axes[2], method='ywm')
axes[2].set_title('PACF → lag=2后截尾→AR(2)✅')
plt.tight_layout()
plt.savefig('/Users/macbookpro/Desktop/股/day30/02_AR2模拟.png', dpi=150, bbox_inches='tight')
plt.show()

print('📊 PACF 在 lag=2 之后全部在蓝色区域内 → AR(2)')

In [ ]:
# 用 AIC 选最优阶数
best_aic = np.inf
best_order = 0
for lag in range(1, 11):
    try:
        m = AutoReg(x_ar2, lags=lag, trend='n').fit()
        aic = m.aic
        if aic < best_aic:
            best_aic = aic
            best_order = lag
    except:
        pass

print(f'AIC 最优阶数: AR({best_order})')
print(f'真实阶数:     AR(2)')
print()

# 拟合最优模型
final = AutoReg(x_ar2, lags=best_order, trend='n').fit()
print(f'估计参数: {final.params}')
print(f'真实参数: {phi}')
print()
print('💡 AIC 的原理：惩罚参数过多 → 在拟合度和简洁性之间找平衡')

---
## 第二部分：MA(q) — 移动平均模型

### 核心公式
$$x_t = \mu + \varepsilon_t + \theta_1\varepsilon_{t-1} + ... + \theta_q\varepsilon_{t-q}$$

**直觉**：今天 = 均值 + 今天的冲击 + 过去 q 天冲击的余波。

**与 AR 的区别**：AR 用过去的**观测值**，MA 用过去的**预测误差**。

**关键**：ACF 在 q 阶后截尾 → 用来判断 MA 的阶数 q。（和 AR 反过来！）

### 2.1 模拟 MA(1)：θ=0.6

In [ ]:
np.random.seed(123)
n = 500
theta1 = 0.6

ar = np.r_[1, 0.]    # AR 部分为空
ma = np.r_[1, theta1]
x_ma1 = sm.tsa.arma_generate_sample(ar=ar, ma=ma, nsample=n, burnin=100)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(x_ma1, linewidth=0.5)
axes[0].set_title(f'MA(1) θ={theta1}'); axes[0].grid(True, alpha=0.3)
plot_acf(x_ma1, lags=30, ax=axes[1])
axes[1].set_title('ACF → lag=1后截尾→MA(1)✅')
plot_pacf(x_ma1, lags=30, ax=axes[2], method='ywm')
axes[2].set_title('PACF（缓慢衰减=MA特征）')
plt.tight_layout()
plt.savefig('/Users/macbookpro/Desktop/股/day30/03_MA1模拟.png', dpi=150, bbox_inches='tight')
plt.show()

print('📊 ACF 在 lag=1 显著，之后截尾 → MA(1)')
print('   注意：AR→看PACF截尾，MA→看ACF截尾，刚好相反！')

### 2.2 AR vs MA 对比速查表

| 特性 | AR(p) | MA(q) |
|------|-------|-------|
| 依赖什么 | 过去的**观测值** | 过去的**预测误差** |
| ACF 特征 | 缓慢衰减（拖尾） | **q阶后截尾** ✅ |
| PACF 特征 | **p阶后截尾** ✅ | 缓慢衰减（拖尾） |
| 类比 | 「今天像昨天」 | 「冲击的余波」 |
| 记忆 | 长期记忆（衰减慢） | 短期记忆（q天就忘） |

> 💡 记忆口诀：**A**CF 截尾 → M**A**；P**A**CF 截尾 → **A**R。

### 2.3 模拟 MA(3)：θ=[0.3, 0.2, 0.1]

三个冲击余波，依次衰减。

In [ ]:
np.random.seed(456)
n = 500
theta = np.array([0.3, 0.2, 0.1])

ar = np.r_[1, 0.]
ma = np.r_[1, theta]
x_ma3 = sm.tsa.arma_generate_sample(ar=ar, ma=ma, nsample=n, burnin=100)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(x_ma3, linewidth=0.5)
axes[0].set_title('MA(3) θ=[0.3, 0.2, 0.1]'); axes[0].grid(True, alpha=0.3)
plot_acf(x_ma3, lags=30, ax=axes[1])
axes[1].set_title('ACF → lag=3后截尾→MA(3)✅')
plot_pacf(x_ma3, lags=30, ax=axes[2], method='ywm')
axes[2].set_title('PACF（拖尾）')
plt.tight_layout()
plt.savefig('/Users/macbookpro/Desktop/股/day30/04_MA3模拟.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 第三部分：ARMA(p,q) — 自回归移动平均

AR + MA 的组合拳。AR 捕捉市场参与者的动量/均值回归行为，MA 捕捉外部冲击。

### 核心观点
> ARMA 通常比单独的 AR 或 MA 用更少的参数，因为两者在参数上冗余。

### 如何选 p 和 q？
**AIC 网格搜索**：遍历 (p,q) 组合，选 AIC 最小的。

### 3.1 模拟 ARMA(2,2) 并恢复参数

In [ ]:
np.random.seed(789)
n = 2000

# 真实参数
phi_true = np.array([0.5, -0.25])
theta_true = np.array([0.5, -0.3])

ar = np.r_[1, -phi_true]
ma = np.r_[1, theta_true]
x_arma22 = sm.tsa.arma_generate_sample(ar=ar, ma=ma, nsample=n, burnin=500)

# AIC 网格搜索
best_aic = np.inf
best_order = (0, 0)
results = []

for p in range(4):
    for q in range(4):
        try:
            m = ARIMA(x_arma22, order=(p, 0, q), trend='n').fit()
            aic = m.aic
            results.append((p, q, aic))
            if aic < best_aic:
                best_aic = aic
                best_order = (p, q)
        except:
            pass

print(f'真实阶数: ARMA(2,2)')
print(f'AIC最优:  ARMA{best_order}')
print()

# 展示 AIC 热力图
aic_grid = np.full((4, 4), np.nan)
for p, q, aic in results:
    aic_grid[p, q] = aic

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(aic_grid, cmap='RdYlGn_r', aspect='auto')
for p in range(4):
    for q in range(4):
        if not np.isnan(aic_grid[p, q]):
            color = 'white' if (p, q) == best_order else 'black'
            ax.text(q, p, f'{aic_grid[p,q]:.0f}', ha='center', va='center', fontsize=11, color=color, fontweight='bold')
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xlabel('q (MA阶数)'); ax.set_ylabel('p (AR阶数)')
ax.set_title('ARMA(p,q) AIC 热力图（颜色越深=AIC越小=越好）')
plt.colorbar(im, shrink=0.8, label='AIC')
plt.tight_layout()
plt.savefig('/Users/macbookpro/Desktop/股/day30/05_ARMA_AIC搜索.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 模型诊断：残差必须是白噪声！

拟合的终极目标 = 残差看起来像白噪声（无自相关、均值为零）。

In [ ]:
# 拟合最优 ARMA 模型
best_model = ARIMA(x_arma22, order=(best_order[0], 0, best_order[1]), trend='n').fit()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0,0].plot(best_model.resid, linewidth=0.5)
axes[0,0].axhline(0, color='red', linestyle='--', linewidth=0.5)
axes[0,0].set_title('残差序列（应该是随机噪声）')
axes[0,0].grid(True, alpha=0.3)

plot_acf(best_model.resid, lags=30, ax=axes[0,1])
axes[0,1].set_title('残差 ACF（应该全部不显著）')

plot_pacf(best_model.resid, lags=30, ax=axes[1,0], method='ywm')
axes[1,0].set_title('残差 PACF（应该全部不显著）')

axes[1,1].hist(best_model.resid, bins=40, color='steelblue', edgecolor='white', alpha=0.8, density=True)
x_range = np.linspace(-4, 4, 100)
axes[1,1].plot(x_range, stats.norm.pdf(x_range, 0, 1), 'r-', linewidth=2, label='标准正态')
axes[1,1].set_title('残差分布 vs 正态分布'); axes[1,1].legend()
plt.tight_layout()
plt.savefig('/Users/macbookpro/Desktop/股/day30/06_残差诊断.png', dpi=150, bbox_inches='tight')
plt.show()

# Ljung-Box 检验：H0=残差独立（白噪声）
lb_pval = acorr_ljungbox(best_model.resid, lags=[20]).iloc[0, 1]
print(f'Ljung-Box p值 = {lb_pval:.4f} {"→ 残差是白噪声 ✅" if lb_pval > 0.05 else "→ 还有自相关未解释 ⚠️"}')

---
## 第四部分：ARIMA(p,d,q) 在 A 股实战

用招商银行数据。注意：
- **价格是非平稳的** → d=1（一阶差分=收益率后就平稳了）
- **收益率是平稳的** → d=0

Auquan 原版用 SPX，我们用招行。

In [ ]:
# 获取数据
df_cmb = safe_fetch('600036')
price = df_cmb['收盘'].values
returns = df_cmb['收盘'].pct_change().dropna().values

print(f'价格数据: {len(price)} 个交易日')
print(f'收益率数据: {len(returns)} 个交易日')
print(f'收益率均值: {returns.mean()*100:.4f}%')
print(f'收益率标准差: {returns.std()*100:.4f}%')

### 4.1 对收益率拟合 ARIMA（d=0，因为收益率已平稳）

In [ ]:
# AIC 网格搜索 ARIMA(p, 0, q) 对招行收益率
best_aic = np.inf
best_order = None

for p in range(5):
    for q in range(5):
        try:
            m = ARIMA(returns, order=(p, 0, q), trend='n').fit()
            if m.aic < best_aic:
                best_aic = m.aic
                best_order = (p, 0, q)
        except:
            pass

print(f'招行收益率 最优ARIMA: {best_order}')
print(f'AIC: {best_aic:.2f}')

# 拟合最优模型
arima_best = ARIMA(returns, order=best_order, trend='n').fit()
print(arima_best.summary())

In [ ]:
# 残差诊断
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0,0].plot(arima_best.resid, linewidth=0.3)
axes[0,0].set_title(f'招行 ARIMA{best_order} 残差'); axes[0,0].grid(True, alpha=0.3)
plot_acf(arima_best.resid, lags=30, ax=axes[0,1])
axes[0,1].set_title('残差 ACF（注意看有没有显著的柱）')
plot_pacf(arima_best.resid, lags=30, ax=axes[1,0], method='ywm')
axes[1,0].set_title('残差 PACF')
axes[1,1].hist(arima_best.resid, bins=50, color='steelblue', edgecolor='white', alpha=0.8, density=True)
x_range = np.linspace(-0.1, 0.1, 200)
axes[1,1].plot(x_range, stats.norm.pdf(x_range, returns.mean(), returns.std()/4), 'r-', linewidth=2)
axes[1,1].set_title('残差分布（肥尾⚠️ 比正态分布更极端）')
plt.tight_layout()
plt.savefig('/Users/macbookpro/Desktop/股/day30/07_ARIMA残差.png', dpi=150, bbox_inches='tight')
plt.show()

# Ljung-Box
lb_pval = acorr_ljungbox(arima_best.resid, lags=[20]).iloc[0, 1]
jb_pval = jarque_bera(arima_best.resid)[1]
print(f'Ljung-Box p值 = {lb_pval:.4f} {"→ 残差独立 ✅" if lb_pval > 0.05 else "→ 残差仍相关 ⚠️"}')
print(f'Jarque-Bera p值 = {jb_pval:.6f} → 残差{"正态" if jb_pval > 0.1 else "非正态⚠️(肥尾)"}')

**关键发现**：即使 ARIMA 找到了「最优」阶数，残差通常仍然：
1. 自相关基本消除 ✅（Ljung-Box p>0.05）
2. 但分布**不是正态的** ⚠️（肥尾=极端值比正态多）
3. 残差平方仍有自相关 → 这就是 GARCH 要解决的问题！

In [ ]:
# 残差平方的 ACF — 这就是 GARCH 存在的理由
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(arima_best.resid**2, lags=30, ax=axes[0])
axes[0].set_title('残差² ACF（显著的柱 → 波动率聚集 → 需要GARCH）')
plot_pacf(arima_best.resid**2, lags=30, ax=axes[1], method='ywm')
axes[1].set_title('残差² PACF')
plt.tight_layout()
plt.savefig('/Users/macbookpro/Desktop/股/day30/08_残差平方ACF.png', dpi=150, bbox_inches='tight')
plt.show()

print('💡 如果残差平方有显著的ACF → 波动率不是常数 → GARCH来建模波动率！')

---
## 第五部分：GARCH(p,q) — 波动率建模

### 核心公式
$$\sigma_t^2 = \omega + \alpha\varepsilon_{t-1}^2 + \beta\sigma_{t-1}^2$$

| 参数 | 含义 |
|------|------|
| ω (omega) | 基础波动率水平 |
| α (alpha) | 昨天冲击对今天波动的影响（短期） |
| β (beta) | 昨天波动对今天波动的影响（长期/持续性） |
| α+β | 波动率持续性：越接近1 = 波动持续越久 |

**直觉**：大波动 → σ²变大 → 今天也倾向大波动。小波动 → σ²变小 → 持续小波动。

### 5.1 模拟 GARCH(1,1) 并恢复参数

In [ ]:
np.random.seed(2)
n = 5000
omega, alpha, beta = 0.2, 0.5, 0.3

w = np.random.normal(size=n)
eps = np.zeros(n)
sig2 = np.zeros(n)

for t in range(1, n):
    sig2[t] = omega + alpha * eps[t-1]**2 + beta * sig2[t-1]
    eps[t] = w[t] * np.sqrt(sig2[t])

print(f'真实参数: ω={omega}, α={alpha}, β={beta}')
print(f'α+β = {alpha+beta} (<1则稳定)')
print()

# 拟合
garch_sim = arch_model(eps, vol='Garch', p=1, q=1).fit(disp='off')
print(garch_sim.summary())

### 5.2 GARCH 在招行收益率上的实战

In [ ]:
# 对招行收益率拟合 GARCH(1,1)
returns_series = df_cmb['收盘'].pct_change().dropna() * 100  # 转为百分比，帮助数值稳定

garch_fit = arch_model(returns_series, vol='Garch', p=1, q=1, dist='normal').fit(disp='off')
print(garch_fit.summary())

# 提取条件波动率
cond_vol = garch_fit.conditional_volatility

In [ ]:
# 对比：收益率 vs 条件波动率
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

# 收益率
axes[0].plot(returns_series.values, linewidth=0.3, color='steelblue')
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_title('招商银行 日收益率')
axes[0].set_ylabel('%')
axes[0].grid(True, alpha=0.3)

# GARCH 条件波动率
axes[1].fill_between(range(len(cond_vol)), cond_vol.values, 0, color='red', alpha=0.4)
axes[1].set_title('GARCH(1,1) 条件波动率（模型拟合的 σ_t）')
axes[1].set_ylabel('%')
axes[1].grid(True, alpha=0.3)

# 滚动波动率 vs GARCH波动率（最后250天）
roll_vol = returns_series.rolling(20).std()
last = 250
axes[2].plot(roll_vol.values[-last:], linewidth=1.5, label='20日滚动波动率（实际）', alpha=0.7)
axes[2].plot(cond_vol.values[-last:], linewidth=1.5, label='GARCH条件波动率（模型）', alpha=0.7)
axes[2].set_title('滚动波动率 vs GARCH波动率（最后250天）')
axes[2].set_ylabel('%')
axes[2].legend(loc='upper left')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/Users/macbookpro/Desktop/股/day30/09_GARCH波动率.png', dpi=150, bbox_inches='tight')
plt.show()

print('📊 GARCH 条件波动率比简单滚动波动率更"及时"——冲击后立即上升，不需要等20天')

### 5.3 GARCH 预测：未来5天波动率

这是 GARCH 最实用的功能——预测未来波动率。

In [ ]:
# 预测未来10天波动率
forecast = garch_fit.forecast(horizon=10, reindex=False)
fcast_var = forecast.variance.iloc[-1]  # 最后一天的预测
fcast_vol = np.sqrt(fcast_var)  # 标准差

fig, ax = plt.subplots(figsize=(12, 5))
days = range(1, 11)
ax.plot(days, fcast_vol.values, 'ro-', linewidth=2, markersize=8)
ax.fill_between(days, fcast_vol.values*0.8, fcast_vol.values*1.2, alpha=0.2, color='red')
ax.set_title('GARCH 未来10天波动率预测')
ax.set_xlabel('天数'); ax.set_ylabel('预测日波动率 (%)')
ax.grid(True, alpha=0.3)
for i, v in enumerate(fcast_vol.values):
    ax.text(i+1, v+0.02, f'{v:.2f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('/Users/macbookpro/Desktop/股/day30/10_GARCH预测.png', dpi=150, bbox_inches='tight')
plt.show()

print('💡 GARCH 预测的波动率会逐渐收敛到长期均值（均值回归特性）')
print('   所以 GARCH 适合短期（1-10天）波动率预测，不适合长期')

---
## 第六部分：参数实验 — 改参数看效果

**这是今天最重要的部分！** 通过改变参数，建立直觉。

### 实验 1：改 AR 系数 φ 看序列行为

φ 越接近 1，序列越"粘"（今天和昨天越像）；φ 越接近 0，越像白噪声。

In [ ]:
np.random.seed(1)
n = 200
phis = [0.0, 0.5, 0.9, 0.99]

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
for i, phi in enumerate(phis):
    x = np.zeros(n)
    w = np.random.normal(0, 1, n)
    for t in range(1, n):
        x[t] = phi * x[t-1] + w[t]
    axes[i].plot(x, linewidth=0.5)
    axes[i].axhline(0, color='black', linewidth=0.3)
    axes[i].set_title(f'φ={phi}: {"白噪声(无记忆)" if phi==0 else "中等记忆" if phi<0.8 else "强记忆(趋势持续)" if phi<0.95 else "极强记忆(≈随机游走)"}')
    axes[i].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/Users/macbookpro/Desktop/股/day30/11_实验_改phi.png', dpi=150, bbox_inches='tight')
plt.show()

print('📊 关键直觉：')
print('  φ=0.0 → 纯噪音，完全不可预测')
print('  φ=0.5 → 有点趋势但不强')
print('  φ=0.9 → 强趋势，但偶尔还会回头')
print('  φ=0.99 → 几乎不回头的趋势，非常难区分是趋势还是随机游走')

### 实验 2：改 ARIMA 阶数看残差变化

阶数太少 → 残差有结构（欠拟合）。阶数太多 → 模型不稳定（过拟合）。

In [ ]:
# 比较三个模型：ARIMA(0,0,0)=纯噪音, ARIMA(3,0,2)=适中, ARIMA(5,0,5)=过多
orders = [(0, 0, 0), best_order, (5, 0, 5)]
labels = ['ARIMA(0,0,0)=白噪声基准', f'ARIMA{best_order}=AIC最优', 'ARIMA(5,0,5)=参数过多']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for row, (order, label) in enumerate(zip(orders, labels)):
    try:
        m = ARIMA(returns, order=order, trend='n').fit()
        axes[row, 0].plot(m.resid, linewidth=0.3)
        axes[row, 0].set_title(f'{label}\n残差序列'); axes[row, 0].grid(True, alpha=0.3)
        plot_acf(m.resid, lags=30, ax=axes[row, 1])
        axes[row, 1].set_title(f'残差 ACF (AIC={m.aic:.0f})')
        plot_acf(m.resid**2, lags=30, ax=axes[row, 2])
        axes[row, 2].set_title('残差² ACF（波动率聚集）')
    except Exception as e:
        print(f'{label}: 拟合失败 - {e}')
plt.tight_layout()
plt.savefig('/Users/macbookpro/Desktop/股/day30/12_实验_改阶数.png', dpi=150, bbox_inches='tight')
plt.show()

print('📊 观察：')
print('  ARIMA(0,0,0): 残差=收益率本身，ACF可能会有结构')
print('  AIC最优阶数:  残差ACF基本干净，但残差²仍有自相关')
print('  过多阶数:     可能过拟合，参数不显著')

### 实验 3：改 GARCH 参数 α 和 β

α 大 = 对近期冲击反应快。β 大 = 波动率有长期记忆。

In [ ]:
np.random.seed(5)
n = 500
w = np.random.normal(size=n)

configs = [
    (0.05, 0.1, 0.85, '高β(0.85): 波动持续很久，像慢火炖'),
    (0.05, 0.5, 0.3, '高α(0.5): 对冲击反应剧烈，像爆炒'),
    (0.05, 0.85, 0.1, '极高α(0.85): 波动来得快去得也快'),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
for i, (omega, alpha, beta, desc) in enumerate(configs):
    eps = np.zeros(n)
    sig2 = np.ones(n) * omega / (1 - alpha - beta)
    for t in range(1, n):
        sig2[t] = omega + alpha * eps[t-1]**2 + beta * sig2[t-1]
        eps[t] = w[t] * np.sqrt(sig2[t])
    axes[i].fill_between(range(n), np.sqrt(sig2), 0, color='red', alpha=0.4)
    axes[i].plot(eps, linewidth=0.3, color='steelblue', alpha=0.8)
    axes[i].set_title(f'ω={omega}, α={alpha}, β={beta} (α+β={alpha+beta}) | {desc}')
    axes[i].set_ylabel('σ')
    axes[i].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/Users/macbookpro/Desktop/股/day30/13_实验_改GARCH参数.png', dpi=150, bbox_inches='tight')
plt.show()

print('📊 关键直觉：')
print('  β大 → 波动像"惯性"大，今天波动大会持续很多天')
print('  α大 → 波动对新冲击敏感，但衰减快')
print('  α+β → 越接近1，波动率越"持久"（金融数据通常 0.95~0.99）')

---
## 第七部分：总结

### 今天学到了什么

| 概念 | 一句话 |
|------|--------|
| AR(p) | 用过去的自己预测未来，PACF定阶 |
| MA(q) | 用过去的预测误差修正当前，ACF定阶 |
| ARMA(p,q) | AR+MA合并，AIC网格搜索定阶 |
| ARIMA(p,d,q) | d=差分次数，把非平稳变平稳后做ARMA |
| GARCH(p,q) | 对波动率建模，α=短期冲击 β=长期记忆 |
| 模型诊断 | 残差必须是白噪声（Ljung-Box p>0.05） |
| 肥尾问题 | 金融收益率残差永远不是正态的，ARIMA/GARCH也救不了 |

### 三个最重要的直觉

1. **ACF截尾→MA, PACF截尾→AR** — 看图就能猜模型
2. **残差²的ACF有结构→需要GARCH** — 波动率不是常数
3. **金融数据永远有肥尾** — 模型只是近似，极端事件比模型预测的多

### 明天 (Day 31) 预告
- 过拟合的量化危害 + 防范
- ML策略实战（特征工程+多模型对比）
- 模型选择五大陷阱深挖

In [ ]:
print('Day 30 完成 ✅')
print()
print('核心收获：')
print('1. 手动模拟了AR(1)/AR(2)/MA(1)/MA(3)/ARMA(2,2)/GARCH(1,1)')
print('2. 全部参数恢复验证通过')
print('3. 用AIC网格搜索做了ARMA/ARIMA的自动选阶')
print('4. 对招行收益率拟合了ARIMA+GARCH')
print('5. 三组参数实验建立了直觉')